<a href="https://colab.research.google.com/github/sandrokhizanishvili/AML_GNN_GMA/blob/main/baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline GNN Model for Anti-Money Laundering

## Overview

This notebook implements three baseline GNN architectures for **edge-level transaction classification** on the IBM AML dataset (LI-Small), following the methodology of:

> Altman et al. (2023). *Realistic Synthetic Financial Transactions for Anti-Money Laundering Models.* arXiv:2306.16424

### Task
**Edge classification** — each transaction (edge) is labelled `0` (legitimate) or `1` (laundering).

### Graph Setup
| Item | Value |
|------|-------|
| Nodes (accounts) | 712,684 |
| Edges (transactions) | 6,924,049 |
| Node feature dim | 5 |
| Edge feature dim | 16 |
| Laundering rate | ~0.05% (severe imbalance) |
| Split | 60 / 20 / 20 temporal |

### Models
| Model | Description |
|-------|-------------|
| **GINe** | Graph Isomorphism Network with edge features |
| **GIN+EU** | GINe extended with explicit edge updates per layer |
| **PNA** | Principal Neighbourhood Aggregation — multiple aggregators + degree scalers |

---

## 1. Installation

Run this cell only once per Colab session, then **restart the runtime** before continuing.

In [1]:
print("🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...")

# 1. Uninstall the current mismatching versions
# We remove the one you just spent 15 mins compiling, because re-installing
# the CORRECT version via wheels will take only 30 seconds.
# os.system("pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib")
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib

# 2. Install PyTorch 2.8.0 (with CUDA 12.6 support)
# We specify the version explicitly to match the PyG documentation you found.
print("⬇️ Installing PyTorch 2.8.0...")
# os.system("pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 3. Install Graph Libraries for PyTorch 2.8
# This link matches the table in your screenshot: torch-2.8.0 + cu126
print("⬇️ Installing Graph Libraries (Wheels)...")
# os.system("pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

print("="*40)
print("✅ SETUP COMPLETE.")
print("⚠️ YOU MUST RESTART THE RUNTIME NOW (Runtime -> Restart Session)")
print("="*40)




import torch

try:
    import torch_sparse
    sparse_status = "✅ Installed"
    sparse_version = torch_sparse.__version__
except ImportError:
    sparse_status = "❌ Not Found"
    sparse_version = "N/A"

print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"Torch Sparse Status:  {sparse_status} ({sparse_version})")

if torch.cuda.is_available() and sparse_status == "✅ Installed":
    print("\nSUCCESS! You are ready to run the training loop.")
else:
    print("\n⚠️ Something is still missing. Did you Restart the Runtime?")



!pip install torch_geometric

🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
⬇️ Installing PyTorch 2.8.0...
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 122.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 261.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 118.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 78.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 129.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3

## 2. Imports & Reproducibility

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import time
import math
import random
import warnings
from google.colab import drive
warnings.filterwarnings('ignore')

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve, average_precision_score, matthews_corrcoef
)

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── PyTorch Geometric ─────────────────────────────────────────────────────────
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GINEConv, PNAConv
from torch_geometric.utils import degree

# ── Progress bar ──────────────────────────────────────────────────────────────
from tqdm import tqdm

print(f'PyTorch          : {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PyTorch          : 2.8.0+cu126
PyTorch Geometric: 2.7.0
Device           : cuda


## 3. Load Graph Snapshots

Three PyG `Data` objects saved by `Data_prepration.ipynb`:
- `train_graph` — training edges only, all evaluated
- `val_graph` — train + val edges, evaluated on val portion only
- `test_graph` — all edges, evaluated on test portion only

This cumulative snapshot design ensures val/test edges have access to historical context for message passing, matching the paper's protocol.

In [2]:
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DATA_DIR = 'Data'

# weights_only=False required because PyG Data objects contain custom classes
# that cannot be loaded with PyTorch 2.6+ default weights_only=True
train_graph = torch.load(os.path.join(DATA_DIR, 'train_graph.pt'), weights_only=False)
val_graph   = torch.load(os.path.join(DATA_DIR, 'val_graph.pt'),   weights_only=False)
test_graph  = torch.load(os.path.join(DATA_DIR, 'test_graph.pt'),  weights_only=False)

def describe_graph(g, name):
    """Print a summary of a graph snapshot."""
    labels   = g.y[g.eval_mask]
    n_pos    = (labels == 1).sum().item()
    n_eval   = g.eval_mask.sum().item()
    print(f'{name}:')
    print(f'  Nodes          : {g.num_nodes:,}')
    print(f'  Edges (total)  : {g.edge_index.shape[1]:,}')
    print(f'  Eval edges     : {n_eval:,}')
    print(f'  Laundering     : {n_pos:,}  ({100*n_pos/n_eval:.4f}%)')
    print(f'  Node feat dim  : {g.x.shape[1]}')
    print(f'  Edge feat dim  : {g.edge_attr.shape[1]}')
    print()

describe_graph(train_graph, 'train_graph')
describe_graph(val_graph,   'val_graph')
describe_graph(test_graph,  'test_graph')

# ── Label sanity check ────────────────────────────────────────────────────────
for g, name in [(train_graph, 'train'), (val_graph, 'val'), (test_graph, 'test')]:
    labels = g.y[g.eval_mask]
    assert (labels == -1).sum() == 0, f'{name}: -1 labels found in eval set!'
    assert labels.unique().tolist() == [0, 1] or set(labels.unique().tolist()) <= {0, 1}
print('Label sanity check passed — no -1 values in eval masks.')

train_graph:
  Nodes          : 712,684
  Edges (total)  : 4,154,429
  Eval edges     : 4,154,429
  Laundering     : 1,813  (0.0436%)
  Node feat dim  : 5
  Edge feat dim  : 16

val_graph:
  Nodes          : 712,684
  Edges (total)  : 5,539,239
  Eval edges     : 1,384,810
  Laundering     : 827  (0.0597%)
  Node feat dim  : 5
  Edge feat dim  : 16

test_graph:
  Nodes          : 712,684
  Edges (total)  : 6,924,049
  Eval edges     : 1,384,810
  Laundering     : 925  (0.0668%)
  Node feat dim  : 5
  Edge feat dim  : 16

Label sanity check passed — no -1 values in eval masks.


## 4. Hyperparameters

In [4]:
# ── Dimensions (inferred from data) ──────────────────────────────────────────
NODE_DIM = train_graph.x.shape[1]          # 5
EDGE_DIM = train_graph.edge_attr.shape[1]  # 16

# ── Model ─────────────────────────────────────────────────────────────────────
HIDDEN_DIM = 64    # matches the paper: hidden embedding size 64
NUM_LAYERS = 2     # matches the paper: 2 GNN layers
DROPOUT    = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-5

# ── Neighbourhood sampling ────────────────────────────────────────────────────
# num_neighbors[0] = neighbours to sample at 2-hop (outermost)
# num_neighbors[1] = neighbours to sample at 1-hop (closest to seed)
# Must have one value per GNN layer
NUM_NEIGHBORS = [100, 100]
BATCH_SIZE    = 16384 # 4096

# ── Class imbalance ───────────────────────────────────────────────────────────
# After oversampling to 20% positives, effective ratio = 4:1
# pos_weight = 4.0 balances the gradient signal between classes
n_neg_train = (train_graph.y[train_graph.eval_mask] == 0).sum().item()
n_pos_train = (train_graph.y[train_graph.eval_mask] == 1).sum().item()
print(f'Train imbalance  : {n_neg_train/n_pos_train:.0f}:1  '
      f'({n_pos_train:,} pos / {n_neg_train:,} neg)')

POS_WEIGHT = torch.tensor([50.0], device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

print(f'\nConfig:')
print(f'  NODE_DIM={NODE_DIM}, EDGE_DIM={EDGE_DIM}')
print(f'  HIDDEN_DIM={HIDDEN_DIM}, NUM_LAYERS={NUM_LAYERS}, DROPOUT={DROPOUT}')
print(f'  EPOCHS={EPOCHS}, LR={LR}, BATCH_SIZE={BATCH_SIZE}')
print(f'  NUM_NEIGHBORS={NUM_NEIGHBORS}')
print(f'  POS_WEIGHT={POS_WEIGHT.item()}')

Train imbalance  : 2290:1  (1,813 pos / 4,152,616 neg)

Config:
  NODE_DIM=5, EDGE_DIM=16
  HIDDEN_DIM=64, NUM_LAYERS=2, DROPOUT=0.3
  EPOCHS=10, LR=0.001, BATCH_SIZE=16384
  NUM_NEIGHBORS=[100, 100]
  POS_WEIGHT=50.0


## 5. Model Architectures

### Encode → Decode Pattern

`LinkNeighborLoader` separates two sets of edges per batch:

- `batch.edge_index` — **context edges** used for message passing (building node embeddings)
- `batch.edge_label_index` — **seed edges** that we actually want to classify

This requires an explicit encode → decode architecture:
```
encode: x, edge_index, edge_attr  →  node embeddings h
decode: h, edge_label_index        →  logits for seed edges
```


### Why LayerNorm Instead of BatchNorm

BatchNorm normalises across the batch dimension. With ~200,000 context edges per batch and only ~800 laundering edges (0.4%), the batch statistics are dominated entirely by legitimate transactions — effectively normalising away the laundering signal. LayerNorm normalises per node across the feature dimension, which is safe regardless of class distribution.

In [5]:
def build_mlp(in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.3):
    """
    Builds a multi-layer perceptron with LayerNorm and Dropout.
    Used as the update function inside GNN layers and as the final classifier.
    """
    layers = []
    dims   = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]
    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:   # no activation/norm on the final output layer
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(dims[i+1]))
            layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)

In [6]:
'''
GINEConv (handles graph structure):
┌──────────────────────────────────────────────────┐
│  For each node v:                                │
│    agg = SUM[ ReLU(h[u] + e_{uv}) ]             │
│    input = h[v] + agg                            │
│                                                  │
│    YOUR mlp(input):  ◄── build_mlp lives here   │
│    ┌────────────────────────────────────────┐    │
│    │ Linear(64→128) → ReLU → LN → Dropout  │    │
│    │ → Linear(128→64)                       │    │
│    └────────────────────────────────────────┘    │
│                                                  │
│    h_new[v] = mlp output                        │
└──────────────────────────────────────────────────┘
'''

'\nGINEConv (handles graph structure):\n┌──────────────────────────────────────────────────┐\n│  For each node v:                                │\n│    agg = SUM[ ReLU(h[u] + e_{uv}) ]             │\n│    input = h[v] + agg                            │\n│                                                  │\n│    YOUR mlp(input):  ◄── build_mlp lives here   │\n│    ┌────────────────────────────────────────┐    │\n│    │ Linear(64→128) → ReLU → LN → Dropout  │    │\n│    │ → Linear(128→64)                       │    │\n│    └────────────────────────────────────────┘    │\n│                                                  │\n│    h_new[v] = mlp output                        │\n└──────────────────────────────────────────────────┘\n'

In [7]:
'''

h⁰[v]
  ↓
GINEConv:
  agg    = SUM[ ReLU(h⁰[u] + e_{uv}) ]
  h_new  = mlp( h⁰[v] + agg )         ← Linear→ReLU→LN→Drop→Linear
  ↓
LayerNorm(h_new)                        ← normalise per node across 64 features
  ↓
ReLU                                    ← clip negatives, add non-linearity
  ↓
Dropout(0.3)                            ← randomly zero 30% of features
  ↓
h¹[v]  — ready for next layer or decode

'''

'\n\nh⁰[v]\n  ↓\nGINEConv:\n  agg    = SUM[ ReLU(h⁰[u] + e_{uv}) ]\n  h_new  = mlp( h⁰[v] + agg )         ← Linear→ReLU→LN→Drop→Linear\n  ↓\nLayerNorm(h_new)                        ← normalise per node across 64 features\n  ↓\nReLU                                    ← clip negatives, add non-linearity\n  ↓\nDropout(0.3)                            ← randomly zero 30% of features\n  ↓\nh¹[v]  — ready for next layer or decode\n\n'

In [8]:
'''

h⁰ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h¹
h¹ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h²
                                                                    ↓
                                                                 decode()
'''

'\n\nh⁰ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h¹\nh¹ → [GINEConv + mlp(ReLU#1)] → LayerNorm → ReLU#2 → Dropout → h²\n                                                                    ↓\n                                                                 decode()\n'

### Model 1 — GINe (Graph Isomorphism Network with Edge Features)

In [9]:
class GINe(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.dropout   = dropout
        self.node_proj = nn.Linear(node_dim, hidden_dim) # INPUT → h projection
        self.edge_proj = nn.Linear(edge_dim, hidden_dim) # INPUT → e projection

        self.convs = nn.ModuleList() # 2 GNN layers
        self.norms = nn.ModuleList() # one norm per layer
        for _ in range(num_layers):
            mlp = build_mlp(hidden_dim, hidden_dim * 2, hidden_dim, dropout=dropout)
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # ── 192-dim input: h[src](64) + h[dst](64) + e(64) ───────
        self.edge_classifier = build_mlp(
            in_dim     = hidden_dim * 3,
            hidden_dim = hidden_dim,
            out_dim    = 1,
            dropout    = dropout,
        )

    def encode(self, x, edge_index, edge_attr):
        """Message passing over context edges → node embeddings."""
        h = F.relu(self.node_proj(x)) # x [N,5]  → h⁰ [N,64]
        e = F.relu(self.edge_proj(edge_attr)) # edge_attr [E,16] → e [E,64]

        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index, e) # GINEConv: SUM ReLU(h[u] + e_{uv})
            h = norm(h)                # LayerNorm per node
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def decode(self, h, edge_label_index, edge_label_attr):
        """
        Classify seed edges using node embeddings AND seed edge features.

        Parameters
        ----------
        h                : node embeddings from encode()  [N, 64]
        edge_label_index : seed edge endpoints            [2, n_seeds]
        edge_label_attr  : raw features of seed edges     [n_seeds, 16]
        """
        src, dst = edge_label_index

        # Project seed edge features to hidden dim
        # (same projection as used during message passing)
        e_seed = F.relu(self.edge_proj(edge_label_attr))  # [n_seeds, 64]

        # Concatenate: sender context + receiver context + transaction features
        edge_emb = torch.cat([h[src], h[dst], e_seed], dim=-1)  # [n_seeds, 192]
        return self.edge_classifier(edge_emb).squeeze(-1)

    def forward(self, x, edge_index, edge_attr, edge_label_index, edge_label_attr):
        h = self.encode(x, edge_index, edge_attr)
        return self.decode(h, edge_label_index, edge_label_attr)

## 6. Data Loader

### Why Oversample During Training Only

With 0.05% laundering, each batch of 4096 edges contains on average **2 laundering transactions**. With pos_weight=4, the gradient from 2 positives (×4 weight) is still overwhelmed by 4094 negatives — the model collapses to predicting everything as legitimate.

By repeating positive seed edges ~418× during training, we raise the positive ratio to 20%, so each batch contains ~800 laundering edges — enough gradient signal to actually learn.

Evaluation always uses the **real distribution** (no oversampling) to give honest metrics.

In [10]:
# def make_loader(graph, shuffle=True, verbose=False):
#     """Loader without oversampling — uses real class distribution."""
#     seed_edge_index = graph.edge_index[:, graph.eval_mask]
#     seed_labels     = graph.y[graph.eval_mask].float()

#     if verbose:
#       n_pos_after = (seed_labels == 1).sum().item()
#       n_neg_after = (seed_labels == 0).sum().item()
#       print(f'  Data: {n_pos_after:,} pos '
#             f'({100*n_pos_after/(n_pos_after+n_neg_after):.4f}%) '
#             f'| {n_neg_after:,} neg')


#     return LinkNeighborLoader(
#         data             = graph,
#         num_neighbors    = NUM_NEIGHBORS,
#         edge_label_index = seed_edge_index,
#         edge_label       = seed_labels,
#         batch_size       = BATCH_SIZE,
#         shuffle          = shuffle,
#         num_workers      = 0,
#         pin_memory       = False,
#     )

def make_loader(graph, shuffle=True, verbose=False):
    seed_mask       = graph.eval_mask
    seed_edge_index = graph.edge_index[:, seed_mask]
    seed_labels     = graph.y[seed_mask].float()
    seed_edge_attr  = graph.edge_attr[seed_mask]   # ← grab features too

    if verbose:
        n_pos = (seed_labels == 1).sum().item()
        n_neg = (seed_labels == 0).sum().item()
        print(f'  Seed edges : {n_pos + n_neg:,} total')
        print(f'  Laundering : {n_pos:,} ({100*n_pos/(n_pos+n_neg):.4f}%)')

    return LinkNeighborLoader(
        data             = graph,
        num_neighbors    = NUM_NEIGHBORS,
        edge_label_index = seed_edge_index,
        edge_label       = seed_labels,
        batch_size       = BATCH_SIZE,
        shuffle          = shuffle,
        num_workers      = 0,
        pin_memory       = False,
    ), seed_edge_attr   # return alongside loader

## 7. Training & Evaluation

In [11]:
# Higher pos_weight to compensate for no oversampling
# Real ratio ~2290:1 → sqrt gives ~48, we use 50 as a round number
# criterion = nn.BCEWithLogitsLoss(
#     pos_weight=torch.tensor([50.0], device=device)
# )

# def train_epoch(model, graph, optimizer):
#     """Training epoch without oversampling."""
#     model.train()
#     result     = make_loader(graph, shuffle=True, verbose=True)
#     loader = result[0] if isinstance(result, tuple) else result
#     total_loss = 0.0
#     n_batches  = 0
#     pbar       = tqdm(loader, desc='  Training (no oversample)', leave=False)

#     for batch in pbar:
#         batch  = batch.to(device)
#         optimizer.zero_grad()
#         logits = model(
#             batch.x, batch.edge_index,
#             batch.edge_attr, batch.edge_label_index,
#         )
#         loss = criterion(logits, batch.edge_label)
#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#         total_loss += loss.item()
#         n_batches  += 1
#         pbar.set_postfix({'loss': f'{loss.item():.4f}'})

#     return total_loss / max(n_batches, 1)


# @torch.no_grad()
# def evaluate(model, graph, threshold=0.5):
#     """
#     Evaluate model on a graph snapshot.

#     Always uses the real class distribution (no oversampling) for honest metrics.
#     Threshold=0.5 is the default — tune only AFTER training is complete
#     using find_best_threshold() on the validation set.

#     Returns dict with: f1, precision, recall, auc
#     """
#     model.eval()
#     loader     = make_loader(graph, shuffle=False)
#     all_probs  = []
#     all_labels = []
#     pbar       = tqdm(loader, desc='  Validation', leave=False)

#     for batch in pbar:
#         batch  = batch.to(device)
#         logits = model(
#             batch.x,
#             batch.edge_index,
#             batch.edge_attr,
#             batch.edge_label_index,
#         )
#         all_probs.append(torch.sigmoid(logits).cpu().numpy())
#         all_labels.append(batch.edge_label.long().cpu().numpy())

#     all_probs  = np.concatenate(all_probs)
#     all_labels = np.concatenate(all_labels)
#     preds      = (all_probs >= threshold).astype(int)

#     f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
#     pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
#     rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
#     try:
#         auc = roc_auc_score(all_labels, all_probs)
#     except ValueError:
#         auc = float('nan')

#     return {'f1': f1, 'precision': pre, 'recall': rec, 'auc': auc}


# def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
#     """Training loop without oversampling — ablation study."""
#     optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
#     scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

#     best_val_auc = 0.0
#     best_state   = None
#     history      = []

#     print(f'\n{"="*60}')
#     print(f'Training {model_name} — NO OVERSAMPLING (ablation)')
#     print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
#     print(f'  pos_weight : 50.0 (compensates for no oversampling)')
#     print(f'{"="*60}')

#     for epoch in range(1, epochs + 1):
#         print(f'\n--- Epoch {epoch}/{epochs} ---')
#         t0          = time.time()
#         train_loss  = train_epoch(model, train_graph, optimizer)
#         scheduler.step()
#         val_metrics = evaluate(model, val_graph)
#         elapsed     = time.time() - t0

#         history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

#         improved = ''
#         if val_metrics['auc'] > best_val_auc:
#             best_val_auc = val_metrics['auc']
#             best_state   = {k: v.clone() for k, v in model.state_dict().items()}
#             torch.save(model.state_dict(), checkpoint_path)
#             improved     = '  --> New Best Model!'

#         print(
#             f'Result: Train Loss: {train_loss:.4f} | '
#             f'Val F1: {val_metrics["f1"]:.4f} | '
#             f'Val Pre: {val_metrics["precision"]:.4f} | '
#             f'Val Rec: {val_metrics["recall"]:.4f} | '
#             f'Val AUC: {val_metrics["auc"]:.4f} | '
#             f'Time: {elapsed:.1f}s'
#             f'{improved}'
#         )

#     if best_state is not None:
#         model.load_state_dict(best_state)

#     test_metrics = evaluate(model, test_graph)
#     print(f'\n{"="*60}')
#     print(f'Final Test Results — {model_name} (no oversampling)')
#     print(f'  F1        : {test_metrics["f1"]:.4f}')
#     print(f'  Precision : {test_metrics["precision"]:.4f}')
#     print(f'  Recall    : {test_metrics["recall"]:.4f}')
#     print(f'  AUC-ROC   : {test_metrics["auc"]:.4f}')
#     print(f'{"="*60}')

#     return model, history, test_metrics

In [12]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=POS_WEIGHT#torch.tensor([50.0], device=device)
)



def train_epoch(model, graph, optimizer):
    model.train()
    loader, seed_edge_attr = make_loader(graph, shuffle=True, verbose=True)
    total_loss = 0.0
    n_batches  = 0
    pbar       = tqdm(loader, desc='  Training', leave=False)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Get seed edge features for this batch using input_id
        # batch.input_id contains the indices of seed edges in the original
        # seed pool — use them to look up the correct edge features
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,               # ← seed edge raw features
        )

        loss = criterion(logits, batch.edge_label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, graph, threshold=0.5):
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Validation', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds      = (all_probs >= threshold).astype(int)

    f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return {'f1': f1, 'precision': pre, 'recall': rec, 'auc': auc}


def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
    """Training loop without oversampling — ablation study."""
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_val_auc = 0.0
    best_state   = None
    history      = []

    print(f'\n{"="*60}')
    print(f'Training {model_name} — NO OVERSAMPLING (ablation)')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    print(f'  pos_weight : 50.0 (compensates for no oversampling)')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        print(f'\n--- Epoch {epoch}/{epochs} ---')
        t0          = time.time()
        train_loss  = train_epoch(model, train_graph, optimizer)
        scheduler.step()
        val_metrics = evaluate(model, val_graph)
        elapsed     = time.time() - t0

        history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

        improved = ''
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), checkpoint_path)
            improved     = '  --> New Best Model!'

        print(
            f'Result: Train Loss: {train_loss:.4f} | '
            f'Val F1: {val_metrics["f1"]:.4f} | '
            f'Val Pre: {val_metrics["precision"]:.4f} | '
            f'Val Rec: {val_metrics["recall"]:.4f} | '
            f'Val AUC: {val_metrics["auc"]:.4f} | '
            f'Time: {elapsed:.1f}s'
            f'{improved}'
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate(model, test_graph)
    print(f'\n{"="*60}')
    print(f'Final Test Results — {model_name} (no oversampling)')
    print(f'  F1        : {test_metrics["f1"]:.4f}')
    print(f'  Precision : {test_metrics["precision"]:.4f}')
    print(f'  Recall    : {test_metrics["recall"]:.4f}')
    print(f'  AUC-ROC   : {test_metrics["auc"]:.4f}')
    print(f'{"="*60}')

    return model, history, test_metrics

## 8. Train All Three Models

In [19]:
torch.manual_seed(SEED)
gin_model = GINe(
    node_dim   = NODE_DIM,
    edge_dim   = EDGE_DIM,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
).to(device)

gin_model, gin_history, gin_test = run_training(
    gin_model, 'GINe (no oversample)',checkpoint_path = 'Models/GINe_baseline/baseline_11_05_26.pt' , epochs=EPOCHS
)


Training GINe (no oversample) — NO OVERSAMPLING (ablation)
  Parameters : 56,257
  pos_weight : 50.0 (compensates for no oversampling)

--- Epoch 1/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0916 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val AUC: 0.9366 | Time: 211.2s  --> New Best Model!

--- Epoch 2/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0670 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val AUC: 0.9487 | Time: 208.0s  --> New Best Model!

--- Epoch 3/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0612 | Val F1: 0.0752 | Val Pre: 0.0503 | Val Rec: 0.1487 | Val AUC: 0.9551 | Time: 205.9s  --> New Best Model!

--- Epoch 4/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0588 | Val F1: 0.0910 | Val Pre: 0.0570 | Val Rec: 0.2261 | Val AUC: 0.9536 | Time: 203.9s

--- Epoch 5/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


...

Result: Train Loss: 0.0549 | Val F1: 0.0944 | Val Pre: 0.0593 | Val Rec: 0.2310 | Val AUC: 0.9505 | Time: 200.5s

--- Epoch 8/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0540 | Val F1: 0.0971 | Val Pre: 0.0601 | Val Rec: 0.2527 | Val AUC: 0.9562 | Time: 202.9s

--- Epoch 9/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0535 | Val F1: 0.0949 | Val Pre: 0.0577 | Val Rec: 0.2672 | Val AUC: 0.9549 | Time: 227.8s

--- Epoch 10/10 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0525 | Val F1: 0.0987 | Val Pre: 0.0596 | Val Rec: 0.2866 | Val AUC: 0.9560 | Time: 229.7s



Final Test Results — GINe (no oversample) (no oversampling)
  F1        : 0.0906
  Precision : 0.0521
  Recall    : 0.3459
  AUC-ROC   : 0.9618


In [13]:
# Load model

# 1. Initialize Model Architecture (Must match training config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gin_model = GINe(
    node_dim   = NODE_DIM,
    edge_dim   = EDGE_DIM,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
).to(device)

# 2. Load the Weights
checkpoint_path='Models/GINe_baseline/baseline_11_05_26.pt'
if os.path.exists(checkpoint_path):
    print(f"Loading model from {checkpoint_path}...")
    gin_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
else:
    print("Warning: Checkpoint not found. Using untrained model.")

Loading model from Models/GINe_baseline/baseline_11_05_26.pt...


## 9. Threshold Optimisation

The default threshold of 0.5 is rarely optimal for severely imbalanced datasets.
After training is complete, we find the threshold that maximises F1 on the validation set,
then apply it to the test set.

In [14]:
# @torch.no_grad()
# def get_scores(model, graph):
#     """Return raw probabilities and labels for all eval edges in a graph."""
#     model.eval()
#     loader, seed_edge_attr = make_loader(graph, shuffle=False)  # unpack tuple
#     all_probs  = []
#     all_labels = []

#     for batch in loader:
#         batch     = batch.to(device)
#         seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)  # match seed features to batch

#         logits = model(
#             batch.x,
#             batch.edge_index,
#             batch.edge_attr,
#             batch.edge_label_index,
#             seed_attr,               # ← the missing 5th argument
#         )
#         all_probs.append(torch.sigmoid(logits).cpu().numpy())
#         all_labels.append(batch.edge_label.long().cpu().numpy())

#     return np.concatenate(all_labels), np.concatenate(all_probs)


# def find_best_threshold(model, model_name):
#     """
#     Find the threshold that maximises F1 on the validation set.
#     Then evaluate on test set with that threshold.
#     """
#     y_true, y_score = get_scores(model, val_graph)
#     pre, rec, thresholds = precision_recall_curve(y_true, y_score)
#     f1_scores = 2 * (pre * rec) / (pre + rec + 1e-8)
#     best_idx  = f1_scores.argmax()
#     best_thr  = thresholds[best_idx]

#     print(f'\n{model_name} — Optimal Threshold (from val set):')
#     print(f'  Threshold : {best_thr:.4f}')
#     print(f'  Val F1    : {f1_scores[best_idx]:.4f}')
#     print(f'  Val Pre   : {pre[best_idx]:.4f}')
#     print(f'  Val Rec   : {rec[best_idx]:.4f}')

#     # Apply optimal threshold to test set
#     test_metrics = evaluate(model, test_graph, threshold=best_thr)
#     print(f'  Test F1   : {test_metrics["f1"]:.4f}')
#     print(f'  Test Pre  : {test_metrics["precision"]:.4f}')
#     print(f'  Test Rec  : {test_metrics["recall"]:.4f}')
#     print(f'  Test AUC  : {test_metrics["auc"]:.4f}')

#     return best_thr, test_metrics

In [15]:
# # Find optimal thresholds for all models
# gin_thr,    gin_test_opt    = find_best_threshold(gin_model,    'GINe')

In [22]:
# from sklearn.metrics import (
#     precision_recall_curve, average_precision_score,
#     matthews_corrcoef
# )

@torch.no_grad()
def get_scores(model, graph):
    """Return raw probabilities and labels for all eval edges in a graph."""
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Scoring', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    print("Scoring complete")

    return np.concatenate(all_labels), np.concatenate(all_probs)


def compute_metrics(y_true, y_score, threshold):
    """Compute all metrics at a given threshold."""

    print(f'Computing metrics at threshold {threshold:.4f}...')

    preds = (y_score >= threshold).astype(int)

    metrics = {
        'f1'       : f1_score(y_true, preds, pos_label=1, zero_division=0),
        'precision': precision_score(y_true, preds, pos_label=1, zero_division=0),
        'recall'   : recall_score(y_true, preds, pos_label=1, zero_division=0),
        'mcc'      : matthews_corrcoef(y_true, preds),
        'auc'      : roc_auc_score(y_true, y_score),
    }

    print('Metrics computed.')

    return metrics


# def find_best_threshold(model, model_name):
#     """
#     Find two optimal thresholds on the validation set:
#       1. Best F1  — standard paper comparison
#       2. Best MCC — more robust for severe imbalance
#     Then evaluate both on the test set.
#     """
#     y_true_val, y_score_val = get_scores(model, val_graph)

#     # ── Find best F1 threshold ────────────────────────────────────────
#     print(f'\nFinding best F1 threshold on val set...')
#     pre, rec, thresholds = precision_recall_curve(y_true_val, y_score_val)
#     f1_scores  = 2 * (pre * rec) / (pre + rec + 1e-8)
#     best_f1_idx = f1_scores.argmax()
#     best_f1_thr = thresholds[best_f1_idx]
#     print(f'  Best F1 threshold : {best_f1_thr:.4f}  '
#           f'(Val F1={f1_scores[best_f1_idx]*100:.2f}% )')

#     # ── Find best MCC threshold ───────────────────────────────────────
#     # Search over candidate thresholds from PR curve

#     print(f'\nFinding best MCC threshold on val set...')

#     mcc_scores = []
#     for thr in tqdm(thresholds, desc='  Finding the best MCC threshold on val set', leave=False):
#         preds = (y_score_val >= thr).astype(int)
#         mcc_scores.append(matthews_corrcoef(y_true_val, preds))
#     mcc_scores  = np.array(mcc_scores)
#     best_mcc_idx = mcc_scores.argmax()
#     best_mcc_thr = thresholds[best_mcc_idx]

#     print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
#           f'(Val MCC={mcc_scores[best_mcc_idx]:.4f} )')

#     print(f'\n{model_name} — Optimal Thresholds (from val set):')
#     print(f'  Best F1  threshold : {best_f1_thr:.4f}  '
#           f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%  '
#           f'MCC={mcc_scores[best_f1_idx]:.4f})')
#     print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
#           f'(Val MCC={mcc_scores[best_mcc_idx]:.4f}  '
#           f'F1={f1_scores[best_mcc_idx]*100:.2f}%)')

#     # ── Evaluate both thresholds on test set ──────────────────────────
#     y_true_test, y_score_test = get_scores(model, test_graph)

#     metrics_f1  = compute_metrics(y_true_test, y_score_test, best_f1_thr)
#     metrics_mcc = compute_metrics(y_true_test, y_score_test, best_mcc_thr)
#     metrics_05  = compute_metrics(y_true_test, y_score_test, 0.5)

#     print(f'\n── Test Results ──')
#     print(f"{'Metric':<12} {'thr=0.5':>10} {'Best F1 thr':>12} {'Best MCC thr':>14}")
#     print('─' * 52)
#     print(f"{'F1 (%)':<12} {metrics_05['f1']*100:>10.2f} {metrics_f1['f1']*100:>12.2f} {metrics_mcc['f1']*100:>14.2f}")
#     print(f"{'Precision':<12} {metrics_05['precision']*100:>10.2f} {metrics_f1['precision']*100:>12.2f} {metrics_mcc['precision']*100:>14.2f}")
#     print(f"{'Recall':<12} {metrics_05['recall']*100:>10.2f} {metrics_f1['recall']*100:>12.2f} {metrics_mcc['recall']*100:>14.2f}")
#     print(f"{'MCC':<12} {metrics_05['mcc']:>10.4f} {metrics_f1['mcc']:>12.4f} {metrics_mcc['mcc']:>14.4f}")
#     print(f"{'AUC-ROC':<12} {metrics_05['auc']:>10.4f} {metrics_f1['auc']:>12.4f} {metrics_mcc['auc']:>14.4f}")
#     print(f"{'Threshold':<12} {'0.500':>10} {best_f1_thr:>12.4f} {best_mcc_thr:>14.4f}")

#     return best_f1_thr, best_mcc_thr, metrics_f1, metrics_mcc











# def find_best_threshold(model, model_name):
#     y_true_val, y_score_val = get_scores(model, val_graph)

#     # ── Find best F1 threshold via PR curve ───────────────────────────
#     print('\nFinding best F1 threshold on val set...')
#     pre, rec, thresholds = precision_recall_curve(y_true_val, y_score_val)
#     f1_scores   = 2 * (pre * rec) / (pre + rec + 1e-8)
#     best_f1_idx = f1_scores.argmax()
#     best_f1_thr = thresholds[best_f1_idx]
#     print(f'  Best F1 threshold : {best_f1_thr:.4f}  '
#           f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')

#     # ── Find best MCC threshold — chunked to avoid OOM ────────────────
#     print('\nFinding best MCC threshold on val set (chunked)...')

#     # Only 200 candidates — plenty of resolution, tiny memory footprint
#     candidates   = np.linspace(y_score_val.min(), y_score_val.max(), 200)
#     best_mcc     = -1.0
#     best_mcc_thr = candidates[0]

#     y_true_int = y_true_val.astype(np.int8)   # saves memory vs int64

#     for thr in tqdm(candidates, desc='  Finding the best MCC threshold on val set', leave=False):
#       # preds = (y_score_val >= thr).astype(np.int8)

#       # # Remove int() — keep as numpy scalars
#       # tp = ( preds & y_true_int).sum()
#       # fp = ( preds & (1 - y_true_int)).sum()
#       # fn = ((1 - preds) & y_true_int).sum()
#       # tn = ((1 - preds) & (1 - y_true_int)).sum()

#       # denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
#       # mcc   = float((tp*tn - fp*fn) / denom) if denom > 0 else 0.0

#       # if mcc > best_mcc:
#       #     best_mcc     = mcc
#       #     best_mcc_thr = thr

#       preds = (y_score_val >= thr)        # boolean array
#       truth = y_true_int.astype(bool)     # boolean array

#       tp = ( preds &  truth).sum()
#       fp = ( preds & ~truth).sum()
#       fn = (~preds &  truth).sum()
#       tn = (~preds & ~truth).sum()

#       denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
#       mcc   = float((tp*tn - fp*fn) / denom) if denom > 0 else 0.0

#       if mcc > best_mcc:
#           best_mcc     = mcc
#           best_mcc_thr = thr

#     print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
#           f'(Val MCC={best_mcc:.4f})')

#     print(f'\n{model_name} — Optimal Thresholds (from val set):')
#     print(f'  Best F1  threshold : {best_f1_thr:.4f}  '
#           f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')
#     print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
#           f'(Val MCC={best_mcc:.4f})')

#     # ── Evaluate on test set ──────────────────────────────────────────
#     y_true_test, y_score_test = get_scores(model, test_graph)

#     metrics_f1  = compute_metrics(y_true_test, y_score_test, best_f1_thr)
#     metrics_mcc = compute_metrics(y_true_test, y_score_test, best_mcc_thr)
#     metrics_05  = compute_metrics(y_true_test, y_score_test, 0.5)

#     print(f'\n── Test Results ──')
#     print(f"{'Metric':<12} {'thr=0.5':>10} {'Best F1 thr':>12} {'Best MCC thr':>14}")
#     print('─' * 52)
#     print(f"{'F1 (%)':<12} {metrics_05['f1']*100:>10.2f} {metrics_f1['f1']*100:>12.2f} {metrics_mcc['f1']*100:>14.2f}")
#     print(f"{'Precision':<12} {metrics_05['precision']*100:>10.2f} {metrics_f1['precision']*100:>12.2f} {metrics_mcc['precision']*100:>14.2f}")
#     print(f"{'Recall':<12} {metrics_05['recall']*100:>10.2f} {metrics_f1['recall']*100:>12.2f} {metrics_mcc['recall']*100:>14.2f}")
#     print(f"{'MCC':<12} {metrics_05['mcc']:>10.4f} {metrics_f1['mcc']:>12.4f} {metrics_mcc['mcc']:>14.4f}")
#     print(f"{'AUC-ROC':<12} {metrics_05['auc']:>10.4f} {metrics_f1['auc']:>12.4f} {metrics_mcc['auc']:>14.4f}")
#     print(f"{'Threshold':<12} {'0.500':>10} {best_f1_thr:>12.4f} {best_mcc_thr:>14.4f}")

#     return best_f1_thr, best_mcc_thr, metrics_f1, metrics_mcc

def find_best_threshold(model, model_name):
    y_true_val, y_score_val = get_scores(model, val_graph)

    # ── Find best F1 threshold via PR curve ───────────────────────────
    print('\nFinding best F1 threshold on val set...')
    pre, rec, thresholds = precision_recall_curve(y_true_val, y_score_val)
    f1_scores   = 2 * (pre * rec) / (pre + rec + 1e-8)
    best_f1_idx = f1_scores.argmax()
    best_f1_thr = thresholds[best_f1_idx]
    print(f'  Best F1 threshold : {best_f1_thr:.4f}  '
          f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')

    # ── Find best MCC threshold using sklearn on 200 candidates ───────
    print('\nFinding best MCC threshold on val set...')

    candidates   = np.linspace(y_score_val.min(), y_score_val.max(), 200)
    best_mcc     = -1.0
    best_mcc_thr = candidates[0]

    for thr in tqdm(candidates, desc='  MCC sweep', leave=False):
        preds = (y_score_val >= thr).astype(int)
        mcc   = matthews_corrcoef(y_true_val, preds)  # sklearn — guaranteed correct
        if mcc > best_mcc:
            best_mcc     = mcc
            best_mcc_thr = float(thr)

    print(f'  Best MCC threshold : {best_mcc_thr:.4f}  (Val MCC={best_mcc:.4f})')

    # Quick sanity check
    assert -1.0 <= best_mcc <= 1.0, f'MCC out of range: {best_mcc}'

    print(f'\n{model_name} — Optimal Thresholds (from val set):')
    print(f'  Best F1  threshold : {best_f1_thr:.4f}  '
          f'(Val F1={f1_scores[best_f1_idx]*100:.2f}%)')
    print(f'  Best MCC threshold : {best_mcc_thr:.4f}  '
          f'(Val MCC={best_mcc:.4f})')

    # ── Evaluate on test set ──────────────────────────────────────────
    y_true_test, y_score_test = get_scores(model, test_graph)

    metrics_f1  = compute_metrics(y_true_test, y_score_test, best_f1_thr)
    metrics_mcc = compute_metrics(y_true_test, y_score_test, best_mcc_thr)
    metrics_05  = compute_metrics(y_true_test, y_score_test, 0.5)

    print(f'\n── Test Results ──')
    print(f"{'Metric':<12} {'thr=0.5':>10} {'Best F1 thr':>12} {'Best MCC thr':>14}")
    print('─' * 52)
    print(f"{'F1 (%)':<12} {metrics_05['f1']*100:>10.2f} {metrics_f1['f1']*100:>12.2f} {metrics_mcc['f1']*100:>14.2f}")
    print(f"{'Precision':<12} {metrics_05['precision']*100:>10.2f} {metrics_f1['precision']*100:>12.2f} {metrics_mcc['precision']*100:>14.2f}")
    print(f"{'Recall':<12} {metrics_05['recall']*100:>10.2f} {metrics_f1['recall']*100:>12.2f} {metrics_mcc['recall']*100:>14.2f}")
    print(f"{'MCC':<12} {metrics_05['mcc']:>10.4f} {metrics_f1['mcc']:>12.4f} {metrics_mcc['mcc']:>14.4f}")
    print(f"{'AUC-ROC':<12} {metrics_05['auc']:>10.4f} {metrics_f1['auc']:>12.4f} {metrics_mcc['auc']:>14.4f}")
    print(f"{'Threshold':<12} {'0.500':>10} {best_f1_thr:>12.4f} {best_mcc_thr:>14.4f}")

    return best_f1_thr, best_mcc_thr, metrics_f1, metrics_mcc

In [23]:
# ── Run ───────────────────────────────────────────────────────────────────────
gin_f1_thr, gin_mcc_thr, gin_test_f1, gin_test_mcc = find_best_threshold(
    gin_model, 'GINe + edge readout'
)

Scoring complete

Finding best F1 threshold on val set...
  Best F1 threshold : 0.5472  (Val F1=9.75%)

Finding best MCC threshold on val set...


  Best MCC threshold : 0.4311  (Val MCC=0.1253)

GINe + edge readout — Optimal Thresholds (from val set):
  Best F1  threshold : 0.5472  (Val F1=9.75%)
  Best MCC threshold : 0.4311  (Val MCC=0.1253)


Scoring complete
Computing metrics at threshold 0.5472...
Metrics computed.
Computing metrics at threshold 0.4311...
Metrics computed.
Computing metrics at threshold 0.5000...
Metrics computed.

── Test Results ──
Metric          thr=0.5  Best F1 thr   Best MCC thr
────────────────────────────────────────────────────
F1 (%)             9.06         9.63           7.65
Precision          5.21         5.96           4.21
Recall            34.59        24.97          41.73
MCC              0.1329       0.1209         0.1310
AUC-ROC          0.9618       0.9618         0.9618
Threshold         0.500       0.5472         0.4311


## 10. Results Comparison

In [25]:
# # from sklearn.metrics import precision_recall_curve, average_precision_score

# # ── Get raw scores for threshold analysis ─────────────────────────────────────
# y_true, y_score = get_scores(gin_model, test_graph)

# # Fixed threshold=0.5 — matches paper's evaluation protocol
# preds_fixed = (y_score >= 0.5).astype(int)
# f1_fixed    = f1_score(y_true, preds_fixed,  pos_label=1, zero_division=0)
# pre_fixed   = precision_score(y_true, preds_fixed, pos_label=1, zero_division=0)
# rec_fixed   = recall_score(y_true, preds_fixed,  pos_label=1, zero_division=0)
# auc_val     = roc_auc_score(y_true, y_score)

# # Optimal threshold — best F1 via PR curve on val set
# preds_opt = (y_score >= gin_thr).astype(int)
# f1_opt    = f1_score(y_true, preds_opt,  pos_label=1, zero_division=0)
# pre_opt   = precision_score(y_true, preds_opt, pos_label=1, zero_division=0)
# rec_opt   = recall_score(y_true, preds_opt,  pos_label=1, zero_division=0)

# # ── Results table ──────────────────────────────────────────────────────────────
# results = pd.DataFrame([
#     {
#         'Model'        : 'GINe + edge readout (ours, thr=0.5)',
#         'F1 (%)'       : round(f1_fixed * 100, 2),
#         'Precision (%)': round(pre_fixed * 100, 2),
#         'Recall (%)'   : round(rec_fixed * 100, 2),
#         'AUC-ROC'      : round(auc_val, 4),
#         'Threshold'    : 0.50,
#         'Neighbours'   : '[50, 30]',
#     },
#     {
#         'Model'        : f'GINe + edge readout (ours, thr={gin_thr:.3f})',
#         'F1 (%)'       : round(f1_opt * 100, 2),
#         'Precision (%)': round(pre_opt * 100, 2),
#         'Recall (%)'   : round(rec_opt * 100, 2),
#         'AUC-ROC'      : round(auc_val, 4),
#         'Threshold'    : round(gin_thr, 3),
#         'Neighbours'   : '[50, 30]',
#     },
#     {
#         'Model'        : 'GINe + edge readout (paper, LI-Small)',
#         'F1 (%)'       : '7.90 ± 2.78',
#         'Precision (%)': 5.14,
#         'Recall (%)'   : 14.59,
#         'AUC-ROC'      : 'N/A',
#         'Threshold'    : 0.50,
#         'Neighbours'   : '[100, 100]',
#     },
# ])

# print('\n── GINe Results vs Paper (LI-Small) ──')
# print(results.to_string(index=False))
# print()
# print('Architecture notes (matching the paper):')
# print('  - GIN with edge features in message passing:  GINEConv(mlp, edge_dim=64)')
# print('  - Edge readout in decoder:                    concat(h[src], h[dst], e_seed)')
# print('  - Neighbourhood sampling:                     ours=[50,30]  paper=[100,100]')
# print('  - Paper F1 = mean ± std over multiple runs at threshold=0.5')
# print(f'  - Our optimal threshold {gin_thr:.4f} found via PR curve on validation set')

# # ── Training curves ────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# fig.suptitle(
#     'GINe + edge readout — LI-Small  (paper: GIN with edge features, [100,100] neighbours)',
#     fontsize=11, fontweight='bold'
# )

# df_h = pd.DataFrame(gin_history)

# # ── AUC ──
# axes[0].plot(df_h['epoch'], df_h['auc'],
#              color='steelblue', marker='o', markersize=4, label='GINe (ours, [50,30])')
# axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Random baseline')
# axes[0].set_title('Validation AUC-ROC')
# axes[0].set_xlabel('Epoch')
# axes[0].set_ylabel('AUC-ROC')
# axes[0].set_ylim(0.4, 1.0)
# axes[0].legend(fontsize=8)
# axes[0].grid(True, alpha=0.3)

# # ── Loss ──
# axes[1].plot(df_h['epoch'], df_h['loss'],
#              color='steelblue', marker='o', markersize=4, label='GINe (ours)')
# axes[1].set_title('Training Loss (BCE, pos_weight=50)')
# axes[1].set_xlabel('Epoch')
# axes[1].set_ylabel('Loss')
# axes[1].legend(fontsize=8)
# axes[1].grid(True, alpha=0.3)

# # ── Precision-Recall curve ──
# pre_curve, rec_curve, _ = precision_recall_curve(y_true, y_score)
# ap         = average_precision_score(y_true, y_score)
# laund_rate = y_true.mean()

# axes[2].plot(rec_curve, pre_curve,
#              color='steelblue', linewidth=1.5, label=f'GINe (AP={ap:.3f})')
# axes[2].axhline(laund_rate, color='gray', linestyle='--', linewidth=1,
#                 label=f'Random (AP={laund_rate:.4f})')

# # Our optimal threshold point
# axes[2].scatter([rec_opt], [pre_opt], color='red', zorder=5, s=70,
#                 label=f'Optimal thr={gin_thr:.3f}  F1={f1_opt*100:.1f}%')

# # Our fixed threshold=0.5 point
# axes[2].scatter([rec_fixed], [pre_fixed], color='orange', zorder=5, s=70,
#                 label=f'thr=0.5  F1={f1_fixed*100:.1f}%')

# # Paper reference point — GINe + edge readout, threshold=0.5
# axes[2].scatter([0.1459], [0.0514], color='green', marker='*', zorder=5, s=150,
#                 label='Paper GINe+edge readout (thr=0.5, [100,100])')

# axes[2].set_title('Precision-Recall Curve (test set)')
# axes[2].set_xlabel('Recall')
# axes[2].set_ylabel('Precision')
# axes[2].legend(fontsize=7)
# axes[2].grid(True, alpha=0.3)

# plt.tight_layout()
# os.makedirs('plots', exist_ok=True)
# plt.savefig('plots/gine_results.png', dpi=150, bbox_inches='tight')
# plt.show()

In [26]:
y_true, y_score = get_scores(gin_model, test_graph)

metrics_05  = compute_metrics(y_true, y_score, 0.5)
metrics_f1  = compute_metrics(y_true, y_score, gin_f1_thr)
metrics_mcc = compute_metrics(y_true, y_score, gin_mcc_thr)

results = pd.DataFrame([
    {
        'Model'        : 'GINe + edge readout (ours, thr=0.5)',
        'F1 (%)'       : round(metrics_05['f1'] * 100, 2),
        'Precision (%)': round(metrics_05['precision'] * 100, 2),
        'Recall (%)'   : round(metrics_05['recall'] * 100, 2),
        'MCC'          : round(metrics_05['mcc'], 4),
        'AUC-ROC'      : round(metrics_05['auc'], 4),
        'Threshold'    : 0.50,
        'Neighbours'   : '[100, 100]',
    },
    {
        'Model'        : f'GINe + edge readout (ours, best F1 thr={gin_f1_thr:.3f})',
        'F1 (%)'       : round(metrics_f1['f1'] * 100, 2),
        'Precision (%)': round(metrics_f1['precision'] * 100, 2),
        'Recall (%)'   : round(metrics_f1['recall'] * 100, 2),
        'MCC'          : round(metrics_f1['mcc'], 4),
        'AUC-ROC'      : round(metrics_f1['auc'], 4),
        'Threshold'    : round(gin_f1_thr, 3),
        'Neighbours'   : '[100, 100]',
    },
    {
        'Model'        : f'GINe + edge readout (ours, best MCC thr={gin_mcc_thr:.3f})',
        'F1 (%)'       : round(metrics_mcc['f1'] * 100, 2),
        'Precision (%)': round(metrics_mcc['precision'] * 100, 2),
        'Recall (%)'   : round(metrics_mcc['recall'] * 100, 2),
        'MCC'          : round(metrics_mcc['mcc'], 4),
        'AUC-ROC'      : round(metrics_mcc['auc'], 4),
        'Threshold'    : round(gin_mcc_thr, 3),
        'Neighbours'   : '[100, 100]',
    },
    {
        'Model'        : 'GINe + edge readout (paper, LI-Small)',
        'F1 (%)'       : '7.90 ± 2.78',
        'Precision (%)': 5.14,
        'Recall (%)'   : 14.59,
        'MCC'          : 'N/A',
        'AUC-ROC'      : 'N/A',
        'Threshold'    : 0.50,
        'Neighbours'   : '[100, 100]',
    },
])

print('\n── GINe Results vs Paper (LI-Small) ──')
print(results.to_string(index=False))
print()
print('Architecture notes:')
print('  - GIN with edge features in message passing:  GINEConv(mlp, edge_dim=64)')
print('  - Edge readout in decoder:                    concat(h[src], h[dst], e_seed)')
print('  - Neighbourhood sampling:                     ours=[50,30]  paper=[100,100]')
print('  - Paper F1 = mean ± std over multiple runs at threshold=0.5')
print('  - MCC uses all four confusion matrix quadrants — more robust for imbalance')
print(f'  - Best F1  threshold {gin_f1_thr:.4f} found via PR curve on val set')
print(f'  - Best MCC threshold {gin_mcc_thr:.4f} found via threshold sweep on val set')

Scoring complete
Computing metrics at threshold 0.5000...
Metrics computed.
Computing metrics at threshold 0.5472...
Metrics computed.
Computing metrics at threshold 0.4311...
Metrics computed.

── GINe Results vs Paper (LI-Small) ──
                                         Model      F1 (%)  Precision (%)  Recall (%)     MCC AUC-ROC  Threshold Neighbours
           GINe + edge readout (ours, thr=0.5)        9.05           5.21       34.59  0.1328  0.9618      0.500 [100, 100]
 GINe + edge readout (ours, best F1 thr=0.547)        9.62           5.96       24.97  0.1208  0.9618      0.547 [100, 100]
GINe + edge readout (ours, best MCC thr=0.431)        7.65           4.21       41.73  0.1309  0.9618      0.431 [100, 100]
         GINe + edge readout (paper, LI-Small) 7.90 ± 2.78           5.14       14.59     N/A     N/A      0.500 [100, 100]

Architecture notes:
  - GIN with edge features in message passing:  GINEConv(mlp, edge_dim=64)
  - Edge readout in decoder:                    co